# RUKOPYS dataset exploration

Minimal reproducible exploration notebook for `UkrainianCatholicUniversity/rukopys`.

Scope:
- load the dataset through HuggingFace `datasets`;
- inspect available splits and feature schema;
- visualize sample images;
- visualize available annotations and bounding boxes;
- compute basic dataset statistics;
- no model training.

In [ ]:
import os
from collections import Counter, defaultdict
from pathlib import Path
from pprint import pprint

PROJECT_ROOT = Path.cwd()
MPLCONFIGDIR = PROJECT_ROOT / "outputs" / "matplotlib"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import DatasetDict, get_dataset_config_names, get_dataset_split_names, load_dataset
from PIL import Image, ImageDraw

DATASET_ID = "UkrainianCatholicUniversity/rukopys"
SEED = 42
MAX_STATS_EXAMPLES_PER_SPLIT = None  # set to a small int for faster smoke tests

EXPECTED_SITE_SPLITS = pd.DataFrame([
    {
        "split": "train",
        "images": 1330,
        "gt_regions": 25523,
        "annotation_source": "annotator / volunteer",
        "description": "Human-annotated; full bboxes + verified transcription",
    },
    {
        "split": "silver",
        "images": 8207,
        "gt_regions": 161065,
        "annotation_source": "auto",
        "description": "Auto-annotated by Qwen3-VL 8B + Gemini; for self-training",
    },
    {
        "split": "test",
        "images": 386,
        "gt_regions": None,
        "annotation_source": None,
        "description": "Images only; submit predictions to Kaggle",
    },
    {
        "split": "private_benchmark",
        "images": 21,
        "gt_regions": None,
        "annotation_source": None,
        "description": "Held out during competition; public after online stage closes",
    },
])

np.random.seed(SEED)

## Discover configs and splits

This cell asks HuggingFace for the actual dataset configs and split names. Do not hard-code assumptions about dataset structure.

In [ ]:
try:
    config_names = get_dataset_config_names(DATASET_ID)
except Exception as exc:
    config_names = [None]
    print(f"Could not list configs with get_dataset_config_names: {exc}")

print("Configs:")
pprint(config_names)

split_names_by_config = {}
for config_name in config_names:
    try:
        split_names = get_dataset_split_names(DATASET_ID, config_name) if config_name else get_dataset_split_names(DATASET_ID)
    except Exception as exc:
        split_names = []
        print(f"Could not list splits for config={config_name}: {exc}")
    split_names_by_config[config_name or "default"] = split_names

print("\nSplits by config:")
pprint(split_names_by_config)

print("\nExpected split counts from competition site:")
display(EXPECTED_SITE_SPLITS)

## Load dataset

If the dataset is gated or private, authenticate with `huggingface-cli login` before running this notebook.

In [ ]:
# Current observation with datasets==4.8.5:
# config "full" exposes train/silver/test conceptually, but split discovery/loading fails
# because train and silver metadata files have different feature schemas.
# Use stable configs for baseline exploration and keep the published split counts above
# as an external sanity check.
preferred_configs = ["gt_only", "test"]
available_configs = set(config_names or [])
configs_to_load = [config for config in preferred_configs if config in available_configs]

if not configs_to_load:
    configs_to_load = [
        config_name
        for config_name, split_names in split_names_by_config.items()
        if split_names and config_name != "default"
    ]

if not configs_to_load and config_names == [None]:
    configs_to_load = [None]

print("Configs selected for loading:", configs_to_load)

loaded_splits = {}
for config_name in configs_to_load:
    config_dataset = load_dataset(DATASET_ID, config_name) if config_name else load_dataset(DATASET_ID)
    if not isinstance(config_dataset, DatasetDict):
        config_dataset = DatasetDict({"train": config_dataset})

    for split_name, split_ds in config_dataset.items():
        key = f"{config_name}_{split_name}" if config_name else split_name
        loaded_splits[key] = split_ds

dataset = DatasetDict(loaded_splits)
print(dataset)

## Print annotation schema

In [ ]:
for split_name, split_ds in dataset.items():
    print(f"\n=== {split_name} ===")
    print(f"Rows: {len(split_ds)}")
    print("Features:")
    pprint(split_ds.features)
    if len(split_ds) > 0:
        sample = split_ds[0]
        print("Sample keys:", list(sample.keys()))
        print("Sample value types:")
        pprint({key: type(value).__name__ for key, value in sample.items()})

## Helpers for image and annotation discovery

The helper functions below intentionally handle several likely annotation shapes. If they cannot identify the structure, they print the raw sample keys/schema instead of guessing.

In [ ]:
def find_image_column(sample):
    for key, value in sample.items():
        if isinstance(value, Image.Image):
            return key
    for key, value in sample.items():
        if isinstance(value, dict) and {"path", "bytes"}.intersection(value.keys()):
            return key
    return None


def to_pil_image(value):
    if isinstance(value, Image.Image):
        return value.convert("RGB")
    if isinstance(value, np.ndarray):
        return Image.fromarray(value).convert("RGB")
    raise TypeError(f"Unsupported image value type: {type(value)}")


def looks_like_bbox(value):
    return (
        isinstance(value, (list, tuple))
        and len(value) == 4
        and all(isinstance(x, (int, float, np.integer, np.floating)) for x in value)
    )


def normalize_region(region):
    if not isinstance(region, dict):
        return None

    bbox = None
    for bbox_key in ("bbox", "box", "bounding_box", "bounds"):
        if bbox_key in region and looks_like_bbox(region[bbox_key]):
            bbox = [float(x) for x in region[bbox_key]]
            break

    if bbox is None:
        return None

    region_type = region.get("type") or region.get("label") or region.get("class") or region.get("category")
    text = region.get("text") or region.get("transcription") or region.get("value") or ""
    return {"bbox": bbox, "type": region_type, "text": text, "raw": region}


def extract_regions(sample):
    for key in ("regions", "annotations", "objects", "bboxes"):
        value = sample.get(key)
        if isinstance(value, list):
            regions = [normalize_region(item) for item in value]
            regions = [item for item in regions if item is not None]
            if regions:
                return key, regions

    for key, value in sample.items():
        if isinstance(value, list) and value and isinstance(value[0], dict):
            regions = [normalize_region(item) for item in value]
            regions = [item for item in regions if item is not None]
            if regions:
                return key, regions

    # Column-oriented fallback: bbox list plus optional type/text lists.
    bbox_key = None
    for key, value in sample.items():
        if isinstance(value, list) and value and all(looks_like_bbox(item) for item in value):
            bbox_key = key
            break

    if bbox_key is None:
        return None, []

    bboxes = sample[bbox_key]
    type_values = next((sample[key] for key in ("types", "labels", "classes", "categories") if isinstance(sample.get(key), list) and len(sample[key]) == len(bboxes)), None)
    text_values = next((sample[key] for key in ("texts", "transcriptions", "values") if isinstance(sample.get(key), list) and len(sample[key]) == len(bboxes)), None)

    regions = []
    for idx, bbox in enumerate(bboxes):
        regions.append({
            "bbox": [float(x) for x in bbox],
            "type": type_values[idx] if type_values is not None else None,
            "text": text_values[idx] if text_values is not None else "",
            "raw": None,
        })
    return bbox_key, regions


first_split_name = next(iter(dataset.keys()))
first_sample = dataset[first_split_name][0]
image_column = find_image_column(first_sample)
annotation_column, first_regions = extract_regions(first_sample)

print("First split:", first_split_name)
print("Detected image column:", image_column)
print("Detected annotation column:", annotation_column)
print("Detected regions in first sample:", len(first_regions))

if image_column is None or annotation_column is None:
    print("\nCould not fully infer image/annotation columns. Inspect raw sample below:")
    pprint(first_sample)

## Visualize sample images and bounding boxes

In [ ]:
def draw_regions(image, regions):
    canvas = image.copy()
    draw = ImageDraw.Draw(canvas)
    for region in regions:
        x1, y1, x2, y2 = region["bbox"]
        label = str(region.get("type") or "")
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        if label:
            draw.text((x1, max(0, y1 - 12)), label, fill="red")
    return canvas


def show_samples(split_name, n=4):
    split_ds = dataset[split_name]
    indices = list(range(min(n, len(split_ds))))
    if not indices:
        print(f"Split {split_name} is empty")
        return

    fig, axes = plt.subplots(len(indices), 2, figsize=(12, 5 * len(indices)))
    if len(indices) == 1:
        axes = np.array([axes])

    for row, idx in enumerate(indices):
        sample = split_ds[idx]
        img_col = find_image_column(sample)
        ann_col, regions = extract_regions(sample)

        if img_col is None:
            print(f"No image column found for {split_name}[{idx}]. Keys: {list(sample.keys())}")
            continue

        image = to_pil_image(sample[img_col])
        axes[row, 0].imshow(image)
        axes[row, 0].set_title(f"{split_name}[{idx}] image")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(draw_regions(image, regions))
        axes[row, 1].set_title(f"{len(regions)} regions from {ann_col}")
        axes[row, 1].axis("off")

    plt.tight_layout()
    plt.show()


show_samples(first_split_name, n=4)

## Basic dataset statistics

In [ ]:
rows = []
type_counts = defaultdict(Counter)
text_lengths = defaultdict(list)
image_sizes = defaultdict(list)

for split_name, split_ds in dataset.items():
    limit = len(split_ds) if MAX_STATS_EXAMPLES_PER_SPLIT is None else min(len(split_ds), MAX_STATS_EXAMPLES_PER_SPLIT)
    region_count = 0
    annotated_images = 0

    for idx in range(limit):
        sample = split_ds[idx]
        img_col = find_image_column(sample)
        if img_col is not None:
            image = to_pil_image(sample[img_col])
            image_sizes[split_name].append(image.size)

        _, regions = extract_regions(sample)
        if regions:
            annotated_images += 1
        region_count += len(regions)

        for region in regions:
            type_counts[split_name][region.get("type")] += 1
            text_lengths[split_name].append(len(str(region.get("text") or "")))

    sizes = image_sizes[split_name]
    widths = [width for width, _ in sizes]
    heights = [height for _, height in sizes]
    lengths = text_lengths[split_name]

    rows.append({
        "split": split_name,
        "examples_scanned": limit,
        "dataset_rows": len(split_ds),
        "annotated_images": annotated_images,
        "regions": region_count,
        "mean_regions_per_scanned_image": region_count / limit if limit else 0,
        "mean_width": float(np.mean(widths)) if widths else None,
        "mean_height": float(np.mean(heights)) if heights else None,
        "mean_text_len": float(np.mean(lengths)) if lengths else None,
        "empty_text_regions": sum(length == 0 for length in lengths),
    })

stats = pd.DataFrame(rows)
display(stats)

stats_for_comparison = stats.copy()
stats_for_comparison["site_split"] = stats_for_comparison["split"].replace({
    "gt_only_train": "train",
    "test_test": "test",
})
comparison = stats_for_comparison.merge(
    EXPECTED_SITE_SPLITS[["split", "images", "gt_regions"]],
    how="left",
    left_on="site_split",
    right_on="split",
    suffixes=("_loaded", "_expected"),
)
display(comparison[["site_split", "split_loaded", "dataset_rows", "images", "regions", "gt_regions"]])

print("Region type counts by split:")
for split_name, counts in type_counts.items():
    print(f"\n{split_name}")
    pprint(dict(counts))